# mIF Pipeline: Prep + Per-Slide InstanSeg/Nimbus

This notebook demonstrates the new workflow for the `instanseg_nimbus` environment:

1. generate channel maps for the selected slide set
2. prepare shared Nimbus normalization JSONs across that slide set
3. run merge, InstanSeg, and Nimbus for one target slide

Nimbus artifacts stay inside each slide's own output folder.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "prototyping").exists():
            return candidate
    raise RuntimeError("Could not find the repository root containing src/ and prototyping/.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"=== {label} ===")
    result = func(*args, **kwargs)
    show(result)
    return result

from mif_pipeline import (
    merge_slide_ometiffs,
    prepare_nimbus_normalization,
    run_instanseg,
    run_nimbus_chunked,
    setup_slides,
)
from mif_pipeline.config import get_slide_config, load_config


In [2]:
CONFIG_PATH = REPO_ROOT / "prototyping" / "prototype_v2-Crop.yaml"
PREP_SLIDE_IDS = ["SLIDE-0329_crop_2048", "SLIDE-0329_crop_2048_2"]
TARGET_SLIDE_ID = PREP_SLIDE_IDS[0]

SETUP_FORCE = False
NIMBUS_PREP_FORCE = False
MERGE_FORCE = False
INSTANSEG_FORCE = False
NIMBUS_FORCE = False

NIMBUS_PREP_CHUNKS = None  # Example: [0, 1]
NIMBUS_RUN_CHUNKS = None   # Example: [0]


In [3]:
config = load_config(CONFIG_PATH)
print("config:", CONFIG_PATH)
print("prep slides:", PREP_SLIDE_IDS)
print("target slide:", TARGET_SLIDE_ID)


config: /home/ratnayn/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml
prep slides: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2']
target slide: SLIDE-0329_crop_2048


## Resolved Slide Paths

These checks should stay simple: each selected slide gets its own `nimbus/chunk_XXX/normalization_dict.json`, its own chunk CSVs, and its own merged `cell_table_full.csv`.

In [5]:
resolved_slides = {slide_id: get_slide_config(config, slide_id) for slide_id in PREP_SLIDE_IDS}
for slide_id, slide in resolved_slides.items():
    nimbus_dir = Path(slide["nimbus"]["output_dir"])
    print(f"[{slide_id}]")
    print("slide_dir:", slide["slide_dir"])
    print("output_dir:", slide["output_dir"])
    print("channel_map_file:", slide["channel_map_file"])
    print("full_merge_path:", slide["full_merge"]["ome_path"])
    print("mask_dir:", slide["mask_export"]["mask_dir"])
    print("nimbus_output_dir:", nimbus_dir)
    print("nimbus_merged_table:", nimbus_dir / "cell_table_full.csv")


[SLIDE-0329_crop_2048]
slide_dir: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048
output_dir: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs
channel_map_file: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/channel_map.generated.json
full_merge_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif
mask_dir: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks
nimbus_output_dir: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus
nimbus_merged_table: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/cell_table_full.csv
[SLIDE-0329_crop_2048_2]
slide_dir: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2
output_dir: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs
channel_map_file: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/channel_map.generated.json
full_merge_pa

## Step 1: Generate Channel Maps

In [6]:
setup_results = stage_call(
    f"setup_slides({PREP_SLIDE_IDS}, force={SETUP_FORCE})",
    setup_slides,
    config,
    slide_ids=PREP_SLIDE_IDS,
    force=SETUP_FORCE,
)


=== setup_slides(['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'], force=False) ===
{
  "slide_ids": [
    "SLIDE-0329_crop_2048",
    "SLIDE-0329_crop_2048_2"
  ],
  "channel_aliases": [
    "R1_CY3_AF",
    "R1_CY5_AF",
    "R1_CY7_AF",
    "R1_FITC_AF",
    "R1_BIT2_RS0584_CY3B",
    "R1_BIT3_RS0015_CY5",
    "R1_BIT4_RS0083_750",
    "R1_DAPI",
    "R1_BIT1_RS0996_488",
    "R2_BIT6_RS0639_CY3B",
    "R2_BIT7_RS0109_CY5",
    "R2_BIT8_RS0255_750",
    "R2_DAPI",
    "R2_BIT5_RS1047_488",
    "R3_BIT10_RS0763_CY3B",
    "R3_BIT11_RS1312_CY5",
    "R3_BIT12_RS0237_750",
    "R3_DAPI",
    "R3_BIT9_RS0805_488",
    "R4_P19_POLYRAT",
    "R4_ANXA10_647",
    "R4_LGALS4_750",
    "R4_DAPI",
    "R4_GFP_POLY_AF488"
  ],
  "validation_passed": true,
  "dry_run": false,
  "slides": [
    {
      "slide_id": "SLIDE-0329_crop_2048",
      "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048",
      "channel_map_output": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipe

## Step 2: Prepare Shared Nimbus Normalization

This computes one normalization dictionary per chunk across the selected slide set and copies each JSON into every slide-local chunk folder.

In [7]:
nimbus_prep_results = stage_call(
    f"prepare_nimbus_normalization({PREP_SLIDE_IDS}, dry_run=False)",
    prepare_nimbus_normalization,
    config,
    PREP_SLIDE_IDS,
    chunk_indices=NIMBUS_PREP_CHUNKS,
    force=NIMBUS_PREP_FORCE,
)


=== prepare_nimbus_normalization(['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'], dry_run=False) ===


/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/utils.py:10: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'].
Channels: ['R1_DAPI'].
Iterate over fovs...
Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'].
Channels: ['R4_GFP_POLY_AF488'].
Iterate over fovs...
Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'].
Channels: ['R4_P19_POLYRAT'].
Iterate over fovs...
Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'].
Channels: ['R4_ANXA10_647'].
Iterate over fovs...
{
  "slide_ids": [
    "SLIDE-0329_crop_2048",
    "SLIDE-0329_crop_2048_2"
  ],
  "chunk_count":

In [8]:
target_slide = get_slide_config(config, TARGET_SLIDE_ID)
chunk_size = int((target_slide.get("nimbus") or {}).get("channel_chunk_size", 1))
nimbus_channels = list((target_slide.get("nimbus") or {}).get("channels") or [])
chunk_aliases = [nimbus_channels[i:i + chunk_size] for i in range(0, len(nimbus_channels), chunk_size)]

print("target nimbus chunks:")
for index, aliases in enumerate(chunk_aliases):
    chunk_dir = Path(target_slide["nimbus"]["output_dir"]) / f"chunk_{index:03d}"
    print({
        "chunk_index": index,
        "aliases": aliases,
        "normalization_dict": str(chunk_dir / "normalization_dict.json"),
        "chunk_table": str(chunk_dir / "nimbus_cell_table.csv"),
    })


target nimbus chunks:
{'chunk_index': 0, 'aliases': ['R1_DAPI'], 'normalization_dict': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_000/normalization_dict.json', 'chunk_table': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_000/nimbus_cell_table.csv'}
{'chunk_index': 1, 'aliases': ['R4_GFP_POLY_AF488'], 'normalization_dict': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_001/normalization_dict.json', 'chunk_table': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_001/nimbus_cell_table.csv'}
{'chunk_index': 2, 'aliases': ['R4_P19_POLYRAT'], 'normalization_dict': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_002/normalization_dict.json', 'chunk_table': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_002/nimbus_cell_table.csv'}
{'chunk_index': 3, 'aliases': ['R4_ANXA10_647'], 

## Step 3: Merge The Target Slide

In [9]:
merge_results = stage_call(
    f"merge_slide_ometiffs({TARGET_SLIDE_ID}, dry_run=False)",
    merge_slide_ometiffs,
    config,
    TARGET_SLIDE_ID,
    force=MERGE_FORCE,
)


=== merge_slide_ometiffs(SLIDE-0329_crop_2048, dry_run=False) ===
[merge] starting full_merge for SLIDE-0329_crop_2048: 24 channels, tile=(512, 512) -> /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif
[merge] writing SLIDE-0329_crop_2048_full.ome.tif: 24 channels, 8 pyramid levels
[merge] channel 1/24 R1_CY3_AF: read level 0
[merge] channel 1/24 R1_CY3_AF: rebuilding pyramid
[merge] channel 1/24 R1_CY3_AF: wrote level 0
[merge] channel 1/24 R1_CY3_AF: wrote pyramid level 1/7
[merge] channel 1/24 R1_CY3_AF: wrote pyramid level 2/7
[merge] channel 1/24 R1_CY3_AF: wrote pyramid level 3/7
[merge] channel 1/24 R1_CY3_AF: wrote pyramid level 4/7
[merge] channel 1/24 R1_CY3_AF: wrote pyramid level 5/7
[merge] channel 1/24 R1_CY3_AF: wrote pyramid level 6/7
[merge] channel 1/24 R1_CY3_AF: wrote pyramid level 7/7
[merge] channel 2/24 R1_CY5_AF: reading level 0
[merge] channel 2/24 R1_CY5_AF: rebuilding pyramid
[merge] channel 2/24 R1_CY5_AF:

## Step 4: Run InstanSeg On The Target Slide

In [10]:
instanseg_results = stage_call(
    f"run_instanseg({TARGET_SLIDE_ID}, dry_run=False)",
    run_instanseg,
    config,
    TARGET_SLIDE_ID,
    force=INSTANSEG_FORCE,
)


=== run_instanseg(SLIDE-0329_crop_2048, dry_run=False) ===
Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/utils/../bioimageio_models/, loading
Requesting default device: cuda
[instanseg] running SLIDE-0329_crop_2048
[instanseg] mode=medium image=/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif pixel_size_um=0.325 tile_size=1000 overlap=100 batch_size=1
[instanseg] prediction_tag=_instanseg_prediction
[instanseg] channels=['R1_DAPI', 'R1_FITC_AF', 'R4_P19_POLYRAT', 'R4_ANXA10_647', 'R4_LGALS4_750'] indices=[7, 3, 19, 20, 21]
[instanseg] eval_kwargs={'resolve_cell_and_nucleus': True, 'cleanup_fragments': True, 'seed_threshold': 0.6}
[instanseg] loaded selected channels from full_merge with pixel_size_um=0.325002437518281; using 0.325 for eval_medium_image(...)


/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/utils/pytorch_utils.py:286: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  sparse_onehot = torch.sparse_coo_tensor(torch.stack((zz, xxyy)).long(), (torch.ones_like(xxyy).float()),
/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/utils/pytorch_utils.py:312: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.

[instanseg] wrote masks cell=/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff nuclear=/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff
{
  "slide_id": "SLIDE-0329_crop_2048",
  "ome_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif",
  "channels": [
    "R1_DAPI",
    "R1_FITC_AF",
    "R4_P19_POLYRAT",
    "R4_ANXA10_647",
    "R4_LGALS4_750"
  ],
  "channel_indices": [
    7,
    3,
    19,
    20,
    21
  ],
  "mode": "medium",
  "model": "fluorescence_nuclei_and_cells",
  "tile_size": 1000,
  "overlap": 100,
  "batch_size": 1,
  "pixel_size_um": 0.325,
  "prediction_tag": "_instanseg_prediction",
  "cell_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff",
  "nuclear_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2

## Step 5: Run Nimbus On The Target Slide

This call uses the slide-local chunk normalization JSONs if they already exist.

In [11]:
nimbus_results = stage_call(
    f"run_nimbus_chunked({TARGET_SLIDE_ID}, dry_run=False)",
    run_nimbus_chunked,
    config,
    TARGET_SLIDE_ID,
    chunk_indices=NIMBUS_RUN_CHUNKS,
    force=NIMBUS_FORCE,
)


=== run_nimbus_chunked(SLIDE-0329_crop_2048, dry_run=False) ===


Detected multi-channel OME-TIFF file: SLIDE-0329_crop_2048_full.ome.tif 
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048_full'].
Channels: ['R1_DAPI'].
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_000
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif...


  0%|          | 0/1 [00:00<?, ?it/s]

Detected multi-channel OME-TIFF file: SLIDE-0329_crop_2048_full.ome.tif 
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048_full'].
Channels: ['R4_GFP_POLY_AF488'].
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_001
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif...


  0%|          | 0/1 [00:00<?, ?it/s]

Detected multi-channel OME-TIFF file: SLIDE-0329_crop_2048_full.ome.tif 
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048_full'].
Channels: ['R4_P19_POLYRAT'].
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_002
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif...


  0%|          | 0/1 [00:00<?, ?it/s]

Detected multi-channel OME-TIFF file: SLIDE-0329_crop_2048_full.ome.tif 
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048_full'].
Channels: ['R4_ANXA10_647'].
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_003
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif...


  0%|          | 0/1 [00:00<?, ?it/s]

{
  "slide_id": "SLIDE-0329_crop_2048",
  "status": "written",
  "full_merge_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif",
  "mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff",
  "output_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus",
  "chunk_count": 4,
  "selected_chunk_indices": [
    0,
    1,
    2,
    3
  ],
  "selected_chunk_count": 4,
  "join_keys": [
    "fov",
    "cell_id"
  ],
  "merged_csv": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/cell_table_full.csv",
  "dry_run": false,
  "chunks": [
    {
      "chunk_index": 0,
      "aliases": [
        "R1_DAPI"
      ],
      "nimbus_channels": [
        "R1_DAPI"
      ],
      "source_paths": [
        "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_1.0.4_R000_DAPI__FINAL_F.ome.tif"
      ],
      "ou

In [12]:
target_nimbus_dir = Path(target_slide["nimbus"]["output_dir"])
print("expected slide-local Nimbus artifacts:")
print("merged cell table:", target_nimbus_dir / "cell_table_full.csv")
for chunk_dir in sorted(target_nimbus_dir.glob("chunk_*")):
    print({
        "chunk_dir": str(chunk_dir),
        "normalization_dict": (chunk_dir / "normalization_dict.json").exists(),
        "chunk_table": (chunk_dir / "nimbus_cell_table.csv").exists(),
    })


expected slide-local Nimbus artifacts:
merged cell table: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/cell_table_full.csv
{'chunk_dir': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_000', 'normalization_dict': True, 'chunk_table': True}
{'chunk_dir': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_001', 'normalization_dict': True, 'chunk_table': True}
{'chunk_dir': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_002', 'normalization_dict': True, 'chunk_table': True}
{'chunk_dir': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/chunk_003', 'normalization_dict': True, 'chunk_table': True}
